In [50]:
# class PlayerObject:
#     def __init__(self):
#         self.reset()

#     def reset(self):
#         self.hp = cfg.MAX_HP
#         self.last_action = None
#         self.action_streak = []
#         self.action_weights = []

#     def get_phase(self):
#         if self.hp_percentage > cfg.PHASE_HIGH_HP:
#             state_phase = "Aggressive"
#         elif self.hp_percentage < cfg.PHASE_LOW_HP:
#             state_phase = "Defensive"
#         else:
#             state_phase = "Balanced"

#         return state_phase

#     def set_weights(self):
#         self.action_weights = cfg.PHASE_WEIGHTS[self.get_phase().lower()].copy()

#     def choose_action_probablistic(self, enemy_action_streak):

#         self.set_weights()

#         # check enemy last state for possible reactions
#         streak = enemy_action_streak[-cfg.REACTIVE_STREAK_LENGTH:]

#         if len(streak) == cfg.REACTIVE_STREAK_LENGTH:
#             if all(a == cfg.ACTION_ATTACK for a in streak):
#                 self.action_weights[cfg.ACTION_DODGE_COUNTER] += cfg.REACTIVE_DODGE_BOOST
#             if all(a == cfg.ACTION_DEFEND for a in streak):
#                 self.action_weights[cfg.ACTION_ATTACK] += cfg.REACTIVE_ATTACK_BOOST

#         total = sum(self.action_weights)
#         self.action_weights = [w / total for w in self.action_weights]

#         return random.choices([cfg.ACTION_ATTACK, cfg.ACTION_DEFEND, cfg.ACTION_DODGE_COUNTER], weights = self.action_weights, k = 1)[0]

#     @property
#     def hp_percentage(self):
#         return max(0.0, self.hp / cfg.MAX_HP)

In [51]:
# class CombatEnv:
#     def __init__(self):
#         self.player = PlayerObject()
#         self.agent = PlayerObject()
#         self.reset()

#     def reset(self):
#         self.player.reset()
#         self.agent.reset()
#         self.turn_count = 0
#         self.done = False
#         return self.get_state()

#     def get_state(self):
#         return {
#             "player_hp_percentage": self.player.hp_percentage,
#             "player_last_action": self.player.last_action,
#             "player_phase": self.player.get_phase(),

#             "agent_hp_percentage": self.agent.hp_percentage,
#             "agent_last_action": self.agent.last_action,
#             "agent_phase": self.agent.get_phase(),
            
#             "turn_count": self.turn_count,
#         }


In [52]:
import config as cfg
from environment import CombatEnv
from player_object import PlayerObject
import random

In [53]:
ce = CombatEnv()
player = PlayerObject()
agent = PlayerObject()

In [ ]:
def _resolve(enemy_action, player_action):
    dmg_to_player = 0
    dmg_to_enemy = 0

    # --- Scenario 1: Both Attack ---
    if enemy_action == cfg.ACTION_ATTACK and player_action == cfg.ACTION_ATTACK:
        
        dmg_to_player += cfg.ATTACK_DAMAGE
        dmg_to_enemy += cfg.ATTACK_DAMAGE

    # --- Scenario 2: Enemy Attacks, Player Defends ---
    elif enemy_action == cfg.ACTION_ATTACK and player_action == cfg.ACTION_DEFEND:

        dmg_to_player += cfg.ATTACK_DAMAGE * cfg.DEFEND_CHIP_DAMAGE_RATIO

    # --- Scenario 3: Player Attacks, Enemy Defends ---
    elif player_action == cfg.ACTION_ATTACK and enemy_action == cfg.ACTION_DEFEND:

        dmg_to_enemy += cfg.ATTACK_DAMAGE * cfg.DEFEND_CHIP_DAMAGE_RATIO

    # --- Scenario 4: Enemy Attacks, Player Dodges ---
    elif enemy_action == cfg.ACTION_ATTACK and player_action == cfg.ACTION_DODGE_COUNTER:

        if random.random() < cfg.DODGE_COUNTER_SUCCESS_RATE:
            dmg_to_enemy += cfg.DODGE_COUNTER_DAMAGE
        else:
            dmg_to_player += cfg.DODGE_COUNTER_FAIL_DAMAGE

    # --- Scenario 5: Player Attacks, Enemy Dodges ---
    elif player_action == cfg.ACTION_ATTACK and enemy_action == cfg.ACTION_DODGE_COUNTER:

        if random.random() < cfg.DODGE_COUNTER_SUCCESS_RATE:
            dmg_to_player += cfg.DODGE_COUNTER_DAMAGE
        else:
            dmg_to_enemy += cfg.DODGE_COUNTER_FAIL_DAMAGE

    return dmg_to_player, dmg_to_enemy

In [87]:
def run_epochs(epochs, print_data = False):
    Track = {"Tie": 0, "Agent Win": 0, "Player Win": 0}
    for i in range (epochs):

        Turns = 0
        ce.reset()

        while ce.player.hp > 0 and ce.agent.hp > 0:

            player_action = ce.player.choose_action_probablistic(ce.agent.action_streak)
            agent_action = ce.agent.choose_action_probablistic(ce.player.action_streak)

            player_dmg, agent_dmg = _resolve(agent_action, player_action)
            ce.player.hp -= player_dmg
            ce.agent.hp -= agent_dmg

            ce.player.hp = max(0, ce.player.hp)
            ce.agent.hp = max(0, ce.agent.hp)

            if print_data:
                print(
                f"--- Turn [{Turns + 1}] Summary ---\n"
                f"Health   : Player -> {ce.player.hp:<8.1f}   |  Agent -> {ce.agent.hp:.1f}\n"
                f"Action   : Player -> {cfg.ACTION_NAMES[player_action]:<8}   |  Agent -> {cfg.ACTION_NAMES[agent_action]}\n"
                f"DmgTaken : Player -> {player_dmg:<8.1f}   |  Agent -> {agent_dmg:.1f}\n"
                f"Phase    : Player -> {ce.player.get_phase()} | Agent -> {ce.agent.get_phase():}\n"
                )

            Turns += 1

        if(ce.player.hp == 0 and ce.agent.hp != 0):

            if print_data:
                print("Agent Wins!")

            Track["Agent Win"] += 1
        elif(ce.player.hp != 0 and ce.agent.hp == 0):

            if print_data:
                print("Player Wins!")

            Track["Player Win"] += 1
        else:

            if print_data:
                print("Both idiots killed eachother")

            Track["Tie"] += 1
    return Track

In [102]:
epochs = 10000
results = run_epochs(epochs, False)

aw = results["Agent Win"]
awr = aw / epochs

pw = results["Player Win"]
pwr = pw / epochs

t = results["Tie"]
tr = t / epochs

print(f"=== Simulation Results ({epochs} Epochs) ===")
print(f"Agent Wins  : {aw:<5} | Win Rate: {awr:>6.1%}")
print(f"Player Wins : {pw:<5} | Win Rate: {pwr:>6.1%}")
print(f"Ties        : {t:<5} | Tie Rate: {tr:>6.1%}")

=== Simulation Results (10000 Epochs) ===
Agent Wins  : 4919  | Win Rate:  49.2%
Player Wins : 4807  | Win Rate:  48.1%
Ties        : 274   | Tie Rate:   2.7%
